In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)
%cd '/content/drive/MyDrive/SportDataChallenge26/2000-2020 SDC'

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Mounted at /content/drive
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1924/1887106070.py", line 3, in <cell line: 0>
    get_ipython().run_line_magic('cd', "'/content/drive/MyDrive/SportDataChallenge26/2000-2020 SDC'")
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During ha

In [ ]:
import pandas as pd
import numpy as np
import torch

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1924/1791582891.py", line 1, in <cell line: 0>
    import pandas as pd
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceb

In [2]:
# Exploring dataset

#df = pd.read_excel("/content/drive/MyDrive/STAT421/Personal_STAT_421_Football/nfl_combine_2000_2020 (1).xlsx")

df = pd.read_csv('nfl_combine_2000_2020.csv')
df = df.dropna(subset=['wAVOE']) # we can't train on missing parts
df.columns

NameError: name 'pd' is not defined

In [ ]:
# FEATURES AND TARGETS

# Within-position z-scores for combine metrics
# This is what the GNN is SUPPOSED to learn, but having it as an explicit
# feature helps all three models and strengthens the engineered baseline
for metric in ['40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle', 'Wt']:
    if metric in df.columns:
        grp = df.groupby('Pos')[metric]
        df[f'{metric}_pos_z'] = (df[metric] - grp.transform('mean')) / (grp.transform('std') + 1e-8)

# Create missing flags for combine metrics
for metric in ['40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle']:
    df[f'{metric}_missing'] = df[metric].isna().astype(int)

# Create log_pick
df['log_pick'] = np.log(df['Pick'])

# Position encoding — same as before
df['Pos_idx'], pos_map = pd.factorize(df['Pos'])
df['pos_enc'] = df['Pos_idx']
pos_idx = df['Pos_idx'].values
num_positions = len(pos_map)

# ── PRIORITY 1 FIX: expanded feature set including combine metrics ────────────
# Original only used ['Rnd', 'Pick', 'Age'] — the gate had nothing
# position-specific to condition on. Now we feed it the combine measurements
# that actually vary by position (40yd matters for WR, Wt matters for OL, etc.)
# feature_cols = ['Rnd', 'Pick', 'Age', '40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle', 'Wt']
COMBINE_FEATS = ['40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle', 'Wt']
COMBINE_Z     = [f'{m}_pos_z' for m in COMBINE_FEATS]
MISSING_FLAGS = [f'{m}_missing' for m in ['40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle']]
DRAFT_FEATS   = ['Rnd', 'Pick', 'log_pick', 'Age']

ALL_FEATS = COMBINE_FEATS + COMBINE_Z + MISSING_FLAGS + DRAFT_FEATS + ['pos_enc']
# Median impute missing combine values (players who skipped events)
# Use median rather than 0 — filling with 0 would make a 0-lb player, etc.
X_raw = df[ALL_FEATS].copy()
X_raw = X_raw.fillna(X_raw.median())
X = X_raw.values

# Lookup table for reference
mapping_df = pd.DataFrame({
    'Code': range(len(pos_map)),
    'Position': pos_map
})
print(mapping_df)

# Regression target — continuous wAVOE
y_reg = df['wAVOE'].values

# Classification target (top 30% = hit)
threshold = df['wAVOE'].quantile(0.7)
df['y_cls'] = (df['wAVOE'] > threshold).astype(int)
y_cls = df['y_cls'].values

# Z-score scale — same formula as before, now across 9 features
X_scaled = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
X = torch.tensor(X_scaled, dtype=torch.float32)
pos_idx = torch.tensor(pos_idx, dtype=torch.long)

print(f"Feature matrix shape: {X.shape}  (should be N x 9)") # this isnt what happened
print(f"Num positions: {num_positions}")

In [ ]:
# Distribution of wAVOE
import matplotlib.pyplot as plt
import seaborn as sns
sns.histplot(df['wAVOE'].dropna(), kde=True, color='skyblue')
plt.title('Distribution of wAVOE (y_reg)')
plt.grid(axis='y', alpha=0.3)

In [ ]:
# Ablation: same GatedModel architecture but replace FiLM with identity
class NoGateModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_positions):
        super().__init__()
        self.fc1      = nn.Linear(input_dim, hidden_dim)
        self.fc2      = nn.Linear(hidden_dim, hidden_dim)
        self.reg_head = nn.Linear(hidden_dim, 1)
        self.cls_head = nn.Linear(hidden_dim, 1)
    def forward(self, x, pos_idx):
        # pos_idx accepted but ignored — no gating
        h = torch.relu(self.fc1(x))
        h = torch.relu(self.fc2(h))
        return self.reg_head(h), self.cls_head(h)

# Train with identical setup as GatedModel
torch.manual_seed(42)
no_gate_model = NoGateModel(input_dim=X.shape[1], hidden_dim=64,
                             num_positions=num_positions)
# ... same training loop ...

In [ ]:
# FiLM Generator — Feature-wise Linear Modulation
# Unchanged from original — architecture is correct, it just needed better inputs

import torch.nn as nn

class FiLMGenerator(nn.Module):
    """
    Generates position-conditioned scale (gamma) and shift (beta) vectors.
    These gate the evaluation network's hidden activations, allowing the model
    to weigh combine metrics differently per position.
    """
    def __init__(self, num_positions, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_positions, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2 * hidden_dim)  # outputs gamma + beta
        )

    def forward(self, pos_idx):
        emb = self.embedding(pos_idx)
        gamma_beta = self.mlp(emb)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        gamma = gamma.clamp(-5, 5)
        beta = beta.clamp(-5, 5)
        return gamma, beta

In [ ]:
# Gated Model (FiLM applied)
# input_dim is now 9 instead of 3 — no other architecture changes needed

class GatedModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_positions):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.film = FiLMGenerator(num_positions, 20, hidden_dim)
        self.reg_head = nn.Linear(hidden_dim, 1)
        self.cls_head = nn.Linear(hidden_dim, 1)

    def forward(self, x, pos_idx):
        gamma, beta = self.film(pos_idx)
        h = self.fc1(x)
        h = gamma * h + beta  # position gate conditions the combine metrics
        h = torch.relu(h)
        h = self.fc2(h)
        h = gamma * h + beta
        h = torch.relu(h)
        reg_out = self.reg_head(h)
        cls_logits = self.cls_head(h)
        return reg_out, cls_logits

In [ ]:
# Loss functions (with class imbalance handling) — unchanged
pos_weight = (y_cls == 0).sum() / (y_cls == 1).sum()
pos_weight = torch.tensor(pos_weight)

loss_fn_reg = nn.MSELoss()
loss_fn_cls = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def compute_loss(reg_pred, reg_true, cls_logits, cls_true):
    return (
        loss_fn_reg(reg_pred, reg_true) +
        loss_fn_cls(cls_logits, cls_true)
    )

In [ ]:
from sklearn.model_selection import train_test_split

# First, split into training+validation and test sets
X_train_val, X_test, pos_train_val, pos_test, yreg_train_val, yreg_test, ycls_train_val, ycls_test = train_test_split(
    X, pos_idx, y_reg, y_cls, test_size=0.2, random_state=42
)

# Then, split training+validation into training and validation sets
X_train, X_val, pos_train, pos_val, yreg_train, yreg_val, ycls_train, ycls_val = train_test_split(
    X_train_val, pos_train_val, yreg_train_val, ycls_train_val, test_size=0.25, random_state=42 # 0.25 of 0.8 is 0.2
)

yreg_train = torch.tensor(yreg_train, dtype=torch.float32).view(-1, 1)
yreg_val   = torch.tensor(yreg_val,   dtype=torch.float32).view(-1, 1)
yreg_test  = torch.tensor(yreg_test,  dtype=torch.float32).view(-1, 1)
ycls_train = torch.tensor(ycls_train, dtype=torch.float32).view(-1, 1)
ycls_val   = torch.tensor(ycls_val,   dtype=torch.float32).view(-1, 1)
ycls_test  = torch.tensor(ycls_test,  dtype=torch.float32).view(-1, 1)

def compute_mae(pred, true):
    return torch.mean(torch.abs(pred - true))

In [ ]:
# Initialize model
#model = GatedModel(input_dim=X.shape[1], hidden_dim=64, num_positions=num_positions)
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
#print(f"GatedModel input_dim: {X.shape[1]}")

In [ ]:
import pandas as pd
from sklearn.metrics import average_precision_score

# 1. Define seeds and storage for results
seeds = [1, 2, 3, 4, 5]  # random seed numbers use to store results for that specific run
gated_metrics_list = [] # empty list to hold results


for seed in seeds:
    # Set the seed for reproducibility before initializing the model
    torch.manual_seed(seed)
    if torch.cuda.is_available(): # use gpu
        torch.cuda.manual_seed_all(seed)

    # Re-initialize the model and optimizer for every seed
    model = GatedModel(input_dim=X.shape[1], hidden_dim=64, num_positions=num_positions)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_mae = float('inf') # initialize best error as infinity, so a lower error will be claimed as best.
    seed_checkpoint_path = f"best_model_seed_{seed}.pt"

    # 4. Training Loop (100 Epochs)
    for epoch in range(100):
        model.train()
        reg_pred, cls_logits = model(X_train, pos_train) # runs through training data to make predictions
        loss = compute_loss(reg_pred, yreg_train, cls_logits, ycls_train) # calculates prediction vs actual results

        optimizer.zero_grad() # reset gradient
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # prevent exploding gradients
        optimizer.step()

        # Validation per Epoch
        model.eval()
        with torch.no_grad(): # stops tracking gradients
            reg_val_pred, _ = model(X_val, pos_val) # makes predictions on validation data
            val_mae = compute_mae(reg_val_pred, yreg_val) # calculates error

        # Save the best checkpoint for THIS specific seed
        if val_mae < best_val_mae: # check if epoch's error is the lowest so far for the seed (prevent overfitting)
            best_val_mae = val_mae
            torch.save(model.state_dict(), seed_checkpoint_path) # save's seed's best performance (certain epoch)

    # 5. Post-Training Evaluation: Load the BEST weights for this seed
    model.load_state_dict(torch.load(seed_checkpoint_path)) # load seed with lowest weights
    model.eval()

    with torch.no_grad():
        # Evaluate on the TEST set
        reg_test_pred, cls_test_logits = model(X_test, pos_test) # final predictions on test using best weights

        # Calculate individual metrics for the TEST set
        mae = compute_mae(reg_test_pred, yreg_test).item() # calculate mae

        probs_test = torch.sigmoid(cls_test_logits) # convert raw model outputs to probabilities of being a hit
        brier = torch.mean((probs_test - ycls_test) ** 2).item() # calculate brier

        pr_auc = average_precision_score(
            ycls_test.detach().cpu().numpy(), # Use ycls_test
            probs_test.detach().cpu().numpy()
        )

        # Store results
        gated_metrics_list.append({
            'Seed': seed,
            'MAE': mae,
            'Brier': brier,
            'PR-AUC': pr_auc
        })
        print(f"Seed {seed} Best Test MAE: {mae:.4f}")

# 6. Aggregate and Display Results
gated_results_df = pd.DataFrame(gated_metrics_list) # aggregate results
print("\n=== FINAL AGGREGATED GATED MODEL TEST RESULTS ===")
summary = gated_results_df.drop(columns=['Seed']).describe().loc[['mean', 'std']]
print(summary)

In [ ]:
# Baseline MLP — unchanged
class MLPBaseline(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.reg_head = nn.Linear(hidden_dim, 1)
        self.cls_head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        h = torch.relu(self.fc2(h))
        return self.reg_head(h), self.cls_head(h)

In [ ]:
# Train Baseline MLP — unchanged
baseline = MLPBaseline(input_dim=X.shape[1], hidden_dim=64)
optimizer_base = torch.optim.Adam(baseline.parameters(), lr=1e-3)
best_val_mae_base = float('inf')

for epoch in range(100):
    baseline.train()
    reg_pred, cls_logits = baseline(X_train)
    loss = compute_loss(reg_pred, yreg_train, cls_logits, ycls_train)
    optimizer_base.zero_grad()
    loss.backward()
    optimizer_base.step()

    baseline.eval()
    with torch.no_grad():
        reg_val_pred, cls_val_logits = baseline(X_val)
        val_mae = compute_mae(reg_val_pred, yreg_val)

    if val_mae < best_val_mae_base:
        best_val_mae_base = val_mae
        torch.save(baseline.state_dict(), 'best_baseline.pt')

    if epoch % 10 == 0:
        print(f"[Baseline] Epoch {epoch}, Val MAE: {val_mae:.4f}")

In [ ]:
# 1. Identify the absolute best seed based on MAE
best_seed_idx = gated_results_df['MAE'].idxmin()
best_seed_val = gated_results_df.loc[best_seed_idx, 'Seed']
mae_gated_best = gated_results_df.loc[best_seed_idx, 'MAE']

print(f"Best Seed found: Seed {int(best_seed_val)} (MAE: {mae_gated_best:.4f})")

# 2. Load those specific weights for the visualizations/plots
model.load_state_dict(torch.load(f"best_model_seed_{int(best_seed_val)}.pt"))
model.eval()

# 3. Final Comparison Variables (using the AVERAGED results for scientific honesty)
mae_gated = summary.loc['mean', 'MAE']
brier_gated = summary.loc['mean', 'Brier']
pr_auc_gated = summary.loc['mean', 'PR-AUC']

In [ ]:
baseline.load_state_dict(torch.load('best_baseline.pt'))
baseline.eval()
with torch.no_grad():
    reg_pred_base_test, cls_logits_base_test = baseline(X_test) # Predict on X_test

mae_base = compute_mae(reg_pred_base_test, yreg_test) # Calculate MAE on yreg_test
probs_base_test = torch.sigmoid(cls_logits_base_test) # Get probabilities for test set
brier_base = torch.mean((probs_base_test - ycls_test) ** 2) # Calculate Brier score on ycls_test

In [ ]:
# PR-AUC — now computed on the test set
from sklearn.metrics import average_precision_score

# pr_auc_gated will be taken from the aggregated results (gated_results_df)
# as it's computed for each seed on the test set and then averaged.
# We still need to compute pr_auc for the baseline model here.
pr_auc_base = average_precision_score(
    ycls_test.detach().cpu().numpy(), # Use ycls_test
    probs_base_test.detach().cpu().numpy()
)

In [ ]:
# Final comparison
print('=== FINAL TEST RESULTS ===')
print('\nGATED MODEL')
print('MAE:   ', mae_gated.item()) # This will now reflect the mean MAE from test sets
print('Brier: ', brier_gated.item()) # This will now reflect the mean Brier from test sets
print('PR-AUC:', pr_auc_gated) # This will now reflect the mean PR-AUC from test sets

print('\nBASELINE MLP')
print('MAE:   ', mae_base.item()) # This will now reflect MAE from test set
print('Brier: ', brier_base.item()) # This will now reflect Brier from test set
print('PR-AUC:', pr_auc_base)

In [ ]:
# ── PRIORITY 3: Gate weight visualization ────────────────────────────────────
# Extracts the average gamma (scale) magnitude that the FiLM gate assigns
# to each position. A high magnitude means the gate is making large adjustments
# for that position — i.e. combine metrics matter more for those positions.

model.eval()
pos_labels = list(pos_map)

gate_gammas = []
with torch.no_grad():
    for i in range(num_positions):
        pid = torch.tensor([i], dtype=torch.long)
        gamma, _ = model.film(pid)
        gate_gammas.append(gamma.squeeze().numpy())

gate_df = pd.DataFrame(gate_gammas, index=pos_labels,
                       columns=[f'h{i}' for i in range(64)])

# Average |gamma| per position — how strongly does each position shift the gate
gate_magnitude = gate_df.abs().mean(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(gate_magnitude.index, gate_magnitude.values, color='steelblue')
ax.set_xlabel('Mean |γ| across hidden units')
ax.set_title('FiLM Gate Activation Magnitude by Position')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig('figures/gate_magnitude.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved figures/gate_magnitude.png")

In [ ]:
# Generate predictions over the full dataset for B's visualizations
# model, X, pos_idx, df are all already defined from training above

model.eval()
with torch.no_grad():
    reg_full, cls_full = model(X, pos_idx)

df['gnn_wavoe_pred'] = reg_full.squeeze().numpy()
df['gnn_hit_prob']   = torch.sigmoid(cls_full).squeeze().numpy()

print(f"Predictions attached — {len(df)} players")
print(df[['Player', 'Pos', 'Pick', 'wAVOE', 'gnn_wavoe_pred', 'gnn_hit_prob']].head())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import os
os.makedirs('figures', exist_ok=True)

# 1. Late-round steal detection scatter
late_rounds = df[df['Pick'] >= 100].copy()

plt.figure(figsize=(12, 7))
sns.scatterplot(data=late_rounds, x='Pick', y='wAVOE',
                hue='gnn_wavoe_pred', palette='viridis',
                size='gnn_wavoe_pred', sizes=(40, 400), alpha=0.7)
plt.axhline(5, color='black', linestyle='--', alpha=0.6)
plt.axvline(150, color='black', linestyle='--', alpha=0.3)
plt.text(155, 12, "STEAL ZONE\n(Model predicted high,\nreality confirmed)",
         bbox=dict(facecolor='white', alpha=0.5), fontweight='bold', fontsize=9)
plt.title("Late-Round Steal Detection: GNN Predicted Value vs. Actual Career Performance")
plt.xlabel("Draft Pick Number (100+)")
plt.ylabel("Actual Career Value (wAVOE)")
plt.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig('figures/steal_detection_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. GNN predicted vs actual — quadrant plot
fig, ax = plt.subplots(figsize=(12, 10))
sns.scatterplot(data=df, x='gnn_wavoe_pred', y='wAVOE',
                hue='Pick', size='St', sizes=(20, 400),
                palette='viridis_r', alpha=0.6, ax=ax)

ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(0, color='black', linestyle='--', linewidth=1)

xmax = df['gnn_wavoe_pred'].quantile(0.97)
xmin = df['gnn_wavoe_pred'].quantile(0.03)
ymax = df['wAVOE'].quantile(0.97)
ymin = df['wAVOE'].quantile(0.03)

ax.text(xmax * 0.6, ymax * 0.8, "THE HITS\n(Model loved, player succeeded)",
        fontsize=9, weight='bold', color='green')
ax.text(xmin * 0.6, ymax * 0.8, "HIDDEN GEMS\n(Model lukewarm, player succeeded)",
        fontsize=9, weight='bold', color='blue')
ax.text(xmin * 0.6, ymin * 0.8, "CORRECT NO-GOs\n(Model hated, player busted)",
        fontsize=9, weight='bold', color='gray')
ax.text(xmax * 0.6, ymin * 0.8, "OVER-HYPED\n(Model loved, player busted)",
        fontsize=9, weight='bold', color='red')

ax.set_title("GNN Accuracy: Predicted Draft Grade vs. Actual NFL Career Value")
ax.set_xlabel("GNN Predicted wAVOE")
ax.set_ylabel("Actual wAVOE")
plt.tight_layout()
plt.savefig('figures/gnn_quadrant_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Combine trait correlation with GNN predictions by position group
traits = ['40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle', 'Wt']

trait_corr = (
    df.groupby('Pos')[traits + ['gnn_wavoe_pred']]
    .corr()['gnn_wavoe_pred']
    .unstack()
    .drop(columns='gnn_wavoe_pred', errors='ignore')
)

plt.figure(figsize=(12, 10))
sns.heatmap(trait_corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title("GNN Scouting DNA: Which Combine Traits Matter Most by Position?")
plt.xlabel("Combine Metric")
plt.ylabel("Position")
plt.tight_layout()
plt.savefig('figures/gnn_trait_corr_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. Top 20 steals table
steals = (
    df[df['Pick'] >= 100]
    .sort_values('gnn_wavoe_pred', ascending=False)
    .head(20)[['Player', 'Pos', 'Pick', 'Season', 'wAVOE', 'gnn_wavoe_pred', 'gnn_hit_prob']]
    .round(3)
)
print("Top 20 GNN-predicted late-round steals:")
print(steals.to_string(index=False))

In [ ]:
import numpy as np
# Save test set predictions to Drive so SDC notebook can load them
np.save('gnn_test_probs.npy', probs_test.detach().cpu().numpy())
np.save('gnn_test_targets.npy', ycls_test.numpy())
print("Saved gnn_test_probs.npy and gnn_test_targets.npy")